In [1]:
import sys, os
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)


In [2]:
from src.models import EarEncoder

enc = EarEncoder(embedding_dim=128, dropout_rate=0.3)
enc.set_phase(1)


Phase 1 — backbone : 0/238 couches entraînables  |  total paramètres entraînables : 1,475,456


In [3]:
enc.summary()

Model: "EarEncoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ear_left            │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ear_right           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ ear_left[0][0],   │
│ (Rescaling)         │ 3)                │            │ ear_right[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb0      │ (None, 7, 7,      │  4,049,571 │ rescaling_2[0][0… │
│ (Functional)        │ 1280)             │            │ rescaling_2[1][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetb0[0… │
│ (GlobalAveragePool… │                   │            │ efficientnetb0[1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 2560)      │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,311,232 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 128)       │          0 │ dense_2[0][0]     │
│ (UnitNormalization) │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,525,027 (21.08 MB)

 Trainable params: 1,475,456 (5.63 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
import numpy as np
import tensorflow as tf
#sanity check : on teste le modèle sur des données factices
# Batch de 4 sujets factices
dummy = {
    'ear_left':  np.random.rand(4, 224, 224, 3).astype('float32'),
    'ear_right': np.random.rand(4, 224, 224, 3).astype('float32'),
}

z = enc.model(dummy, training=False)
print(f'Shape sortie  : {z.shape}')          # → (4, 128)
print(f'Normes L2     : {tf.norm(z, axis=-1).numpy()}')  # → [1. 1. 1. 1.]


Shape sortie  : (4, 128)
Normes L2     : [0.99999994 0.99999994 0.99999994 1.        ]
